In [1]:
import ee
ee.Authenticate()
ee.Initialize()

Enter verification code:  4/1AeoWuM9n84eJcw_4NjWBgMfPes9-B6vC_nmL7lNGbi1rW1fn_s_bD595c64



Successfully saved authorization token.


In [5]:
# ============================================================
# CLASIFICACIÓN MULTITEMPORAL RF CON GEE + PYTHON
# ============================================================
#
# Ejecutar en Jupyter Notebook
#
# Requiere:
# pip install earthengine-api geemap pandas
#
# ============================================================



# ============================================================
# 1. LIBRERÍAS
# ============================================================

import ee
import geemap
import pandas as pd


# ============================================================
# 2. INICIALIZAR EARTH ENGINE
# ============================================================

ee.Initialize()


# ============================================================
# 3. ÁREA DE ESTUDIO
# ============================================================

roi = ee.FeatureCollection(
    'projects/ee-lgalvez75-488016/assets/Localidad_Usme'
)



# ============================================================
# 4. MUESTRAS DE ENTRENAMIENTO
# ============================================================

Natural = (
    ee.FeatureCollection(
        'projects/ee-lgalvez75-488016/assets/Natural'
    )
    .map(lambda f: f.set('class', 3))
)

Urbano = (
    ee.FeatureCollection(
        'projects/ee-lgalvez75-488016/assets/Urbano'
    )
    .map(lambda f: f.set('class', 2))
)

Mineria = (
    ee.FeatureCollection(
        'projects/ee-lgalvez75-488016/assets/Mineria'
    )
    .map(lambda f: f.set('class', 5))
)

Agua = (
    ee.FeatureCollection(
        'projects/ee-lgalvez75-488016/assets/Agua'
    )
    .map(lambda f: f.set('class', 4))
)

Agricola = (
    ee.FeatureCollection(
        'projects/ee-lgalvez75-488016/assets/Agricola'
    )
    .map(lambda f: f.set('class', 1))
)



# ============================================================
# 5. UNIÓN DE MUESTRAS
# ============================================================

training_polygons = (
    Natural
    .merge(Urbano)
    .merge(Mineria)
    .merge(Agua)
    .merge(Agricola)
)



# ============================================================
# 6. LISTA DE AÑOS
# ============================================================

years = [
    1990,
    1995,
    2000,
    2010,
    2015,
    2020,
    2024
]



# ============================================================
# 7. FUNCIÓN LANDSAT
# ============================================================

def mask_landsat(image):

    qa = image.select('QA_PIXEL')

    cloud = qa.bitwiseAnd(1 << 3).eq(0)

    shadow = qa.bitwiseAnd(1 << 4).eq(0)

    return (
        image
        .updateMask(cloud)
        .updateMask(shadow)
    )



# ============================================================
# 8. FUNCIÓN SENTINEL
# ============================================================

def mask_s2(image):

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10

    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return (
        image
        .updateMask(mask)
        .divide(10000)
    )



# ============================================================
# 9. FUNCIÓN CONSTRUIR IMAGEN
# ============================================================

def build_image(year):

    start = ee.Date.fromYMD(year, 1, 1)

    end = ee.Date.fromYMD(year, 12, 31)



    # ========================================================
    # LANDSAT 5
    # ========================================================

    if year <= 2010:

        collection = (
            ee.ImageCollection(
                'LANDSAT/LT05/C02/T1_L2'
            )
            .filterBounds(roi)
            .filterDate(start, end)
            .filter(
                ee.Filter.lt(
                    'CLOUD_COVER',
                    80
                )
            )
            .map(mask_landsat)
        )

        image = collection.median()

        image = image.select(
            [
                'SR_B1',
                'SR_B2',
                'SR_B3',
                'SR_B4',
                'SR_B5',
                'SR_B7'
            ],
            [
                'B2',
                'B3',
                'B4',
                'B8',
                'B11',
                'B12'
            ]
        )

        image = (
            image
            .multiply(0.0000275)
            .add(-0.2)
        )



    # ========================================================
    # LANDSAT 8
    # ========================================================

    elif year <= 2020:

        collection = (
            ee.ImageCollection(
                'LANDSAT/LC08/C02/T1_L2'
            )
            .filterBounds(roi)
            .filterDate(start, end)
            .filter(
                ee.Filter.lt(
                    'CLOUD_COVER',
                    80
                )
            )
            .map(mask_landsat)
        )

        image = collection.median()

        image = image.select(
            [
                'SR_B2',
                'SR_B3',
                'SR_B4',
                'SR_B5',
                'SR_B6',
                'SR_B7'
            ],
            [
                'B2',
                'B3',
                'B4',
                'B8',
                'B11',
                'B12'
            ]
        )

        image = (
            image
            .multiply(0.0000275)
            .add(-0.2)
        )



    # ========================================================
    # SENTINEL 2
    # ========================================================

    else:

        collection = (
            ee.ImageCollection(
                'COPERNICUS/S2_SR_HARMONIZED'
            )
            .filterBounds(roi)
            .filterDate(start, end)
            .filter(
                ee.Filter.lt(
                    'CLOUDY_PIXEL_PERCENTAGE',
                    40
                )
            )
            .map(mask_s2)
        )

        image = collection.median()

        image = image.select([
            'B2',
            'B3',
            'B4',
            'B8',
            'B11',
            'B12'
        ])



    # ========================================================
    # SUAVIZADO
    # ========================================================

    kernel = ee.Kernel.gaussian(
        radius=2,
        sigma=1,
        units='pixels',
        normalize=True
    )

    image = (
        image
        .convolve(kernel)
        .clip(roi)
    )



    # ========================================================
    # NDVI
    # ========================================================

    ndvi = (
        image
        .normalizedDifference(['B8', 'B4'])
        .rename('NDVI')
    )



    # ========================================================
    # NDBI
    # ========================================================

    ndbi = (
        image
        .normalizedDifference(['B11', 'B8'])
        .rename('NDBI')
    )



    # ========================================================
    # BSI
    # ========================================================

    bsi = image.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {
            'SWIR': image.select('B11'),
            'RED': image.select('B4'),
            'NIR': image.select('B8'),
            'BLUE': image.select('B2')
        }
    ).rename('BSI')



    # ========================================================
    # STACK FINAL
    # ========================================================

    return (
        image
        .addBands(ndvi)
        .addBands(ndbi)
        .addBands(bsi)
    )



# ============================================================
# 10. FUNCIÓN CLASIFICAR AÑO
# ============================================================

def classify_year(year):

    print(f'\nProcesando año: {year}')

    image = build_image(year)

    scale = 10 if year >= 2024 else 30



    # ========================================================
    # EXTRAER MUESTRAS
    # ========================================================

    training = image.sampleRegions(
        collection=training_polygons,
        properties=['class'],
        scale=scale,
        geometries=False,
        tileScale=4
    )



    # ========================================================
    # ELIMINAR NULOS
    # ========================================================

    training = training.filter(
        ee.Filter.notNull(
            image.bandNames()
        )
    )



    # ========================================================
    # RANDOM SPLIT
    # ========================================================

    training = training.randomColumn('random')

    train = training.filter(
        ee.Filter.lt('random', 0.7)
    )

    test = training.filter(
        ee.Filter.gte('random', 0.7)
    )



    # ========================================================
    # RANDOM FOREST
    # ========================================================

    classifier = (
        ee.Classifier.smileRandomForest(
            numberOfTrees=200,
            bagFraction=0.7,
            seed=42
        )
        .train(
            features=train,
            classProperty='class',
            inputProperties=image.bandNames()
        )
    )



    # ========================================================
    # CLASIFICACIÓN
    # ========================================================

    classified = image.classify(classifier)



    # ========================================================
    # VALIDACIÓN
    # ========================================================

    validated = test.classify(classifier)

    matrix = validated.errorMatrix(
        'class',
        'classification'
    )

    accuracy = matrix.accuracy().getInfo()

    kappa = matrix.kappa().getInfo()

    print(f'Accuracy {year}: {accuracy}')

    print(f'Kappa {year}: {kappa}')



    # ========================================================
    # CÁLCULO DE ÁREAS
    # ========================================================

    area_image = (
        ee.Image.pixelArea()
        .divide(10000)
        .addBands(classified)
    )



    areas = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(
            groupField=1,
            groupName='class'
        ),
        geometry=roi.geometry(),
        scale=scale,
        maxPixels=1e13
    )



    groups = ee.List(
        areas.get('groups')
    )



    data = groups.getInfo()



    # ========================================================
    # ÁREA TOTAL DEL AÑO
    # ========================================================

    total_area = sum(
        [item['sum'] for item in data]
    )



    # ========================================================
    # CREAR FILAS
    # ========================================================

    rows = []

    for item in data:

        area = item['sum']

        percentage = (
            area / total_area
        ) * 100

        rows.append({
            'year': year,
            'class': item['class'],
            'area_ha': area,
            'percentage': percentage,
            'accuracy': accuracy,
            'kappa': kappa
        })



    return classified, rows



# ============================================================
# 11. EJECUTAR TODOS LOS AÑOS
# ============================================================

all_rows = []

classified_images = {}



for year in years:

    classified, rows = classify_year(year)

    classified_images[year] = classified

    all_rows.extend(rows)



# ============================================================
# 12. CREAR DATAFRAME
# ============================================================

df = pd.DataFrame(all_rows)



# ============================================================
# 13. RENOMBRAR CLASES
# ============================================================

class_names = {
    3: 'Natural',
    2: 'Urbano',
    5: 'Mineria',
    4: 'Agua',
    1: 'Agricola'
}

df['class_name'] = df['class'].map(class_names)



# ============================================================
# 14. REDONDEAR VALORES
# ============================================================

df['area_ha'] = df['area_ha'].round(2)

df['percentage'] = df['percentage'].round(2)

df['accuracy'] = df['accuracy'].round(3)

df['kappa'] = df['kappa'].round(3)



# ============================================================
# 15. ORDENAR COLUMNAS
# ============================================================

df = df[
    [
        'year',
        'class',
        'class_name',
        'area_ha',
        'percentage',
        'accuracy',
        'kappa'
    ]
]



# ============================================================
# 16. MOSTRAR TABLA COMPLETA
# ============================================================

print('\nTABLA COMPLETA\n')

print(df)



# ============================================================
# 17. TABLA PIVOTE DE ÁREAS
# ============================================================

pivot_area = df.pivot_table(
    index='year',
    columns='class_name',
    values='area_ha'
)

print('\nÁREA POR CLASE (ha)\n')

print(pivot_area)



# ============================================================
# 18. TABLA PIVOTE DE PORCENTAJES
# ============================================================

pivot_percentage = df.pivot_table(
    index='year',
    columns='class_name',
    values='percentage'
)

print('\nPORCENTAJE POR CLASE (%)\n')

print(pivot_percentage)


Procesando año: 1990
Accuracy 1990: 0.9792079207920792
Kappa 1990: 0.9686656634995278

Procesando año: 1995
Accuracy 1995: 0.9866446826051113
Kappa 1995: 0.9759784350876944

Procesando año: 2000
Accuracy 2000: 0.9779723991507431
Kappa 2000: 0.9661853779633636

Procesando año: 2010
Accuracy 2010: 0.9942013705851345
Kappa 2010: 0.9889843078164591

Procesando año: 2015
Accuracy 2015: 0.9955969517358171
Kappa 2015: 0.9922863495165417

Procesando año: 2020
Accuracy 2020: 0.9961525593844095
Kappa 2020: 0.9931393310970069

Procesando año: 2024
Accuracy 2024: 0.9912603240886162
Kappa 2024: 0.9844450921407777

TABLA COMPLETA

    year  class class_name   area_ha  percentage  accuracy  kappa
0   1990      1   Agricola   5478.62       47.89     0.979  0.969
1   1990      2     Urbano   1633.95       14.28     0.979  0.969
2   1990      3    Natural   4176.10       36.50     0.979  0.969
3   1990      4       Agua     84.80        0.74     0.979  0.969
4   1990      5    Mineria     66.94        